In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/preprocessed_fake_job_data.csv')

In [4]:
df = df.fillna("")

In [5]:
df.isnull().sum()

,0
title,0
location,0
company_profile,0
description,0
requirements,0
benefits,0
telecommuting,0
has_company_logo,0
has_questions,0
employment_type,0


In [6]:
X = df['clean_text']

In [7]:
y = df['fraudulent']

In [8]:
X.shape

(17880,)

In [9]:
y.shape

(17880,)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=10000)

X = tfidf.fit_transform(X)

In [15]:
df.head(2)

,title,location,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,...,required_education,industry,function,fraudulent,text,text_length,clean_text,missing_company_profile,missing_benefits,short_description
0,Marketing Intern,"US, NY, New York","We're Food52, and we've created a groundbreaki...","Food52, a fast-growing, James Beard Award-winn...",Experience with content management systems a m...,,0,1,0,Other,...,Unknown,Unknown,Marketing,0,marketing intern us ny new york we re food ...,2708,marketing intern us ny new york food created g...,0,1,0
1,Customer Service - Cloud Video Production,"NZ, , Auckland","90 Seconds, the worlds Cloud Video Production ...",Organised - Focused - Vibrant - Awesome!Do you...,What we expect from you:Your key responsibilit...,What you will get from usThrough being part of...,0,1,0,Full-time,...,Unknown,Marketing and Advertising,Customer Service,0,customer service cloud video production nz ...,6217,customer service cloud video production nz auc...,0,0,0


In [12]:
print(X.shape)

(17880, 10000)


In [13]:
print(X[0])

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 185 stored elements and shape (1, 10000)>
  Coords	Values
  (0, 5506)	0.07930989094700609
  (0, 4817)	0.05176307025358108
  (0, 9465)	0.013840725029155048
  (0, 6095)	0.03911864291664189
  (0, 5988)	0.09312827561669589
  (0, 9968)	0.14933352462363034
  (0, 3789)	0.5297179121881432
  (0, 2215)	0.04406621713391454
  (0, 4155)	0.06457176234803845
  (0, 783)	0.0885452315247139
  (0, 9853)	0.08803044482446501
  (0, 2099)	0.25964967302478437
  (0, 8226)	0.034152923511162586
  (0, 8760)	0.07162437379944969
  (0, 1972)	0.09842466053247127
  (0, 1475)	0.06639266889009003
  (0, 4386)	0.10660108648337477
  (0, 2100)	0.14172387748207052
  (0, 4057)	0.03977083738186209
  (0, 3329)	0.04180132970230201
  (0, 5946)	0.028051584105407267
  (0, 6179)	0.023429025623965632
  (0, 6682)	0.03628322164149758
  (0, 9137)	0.03140214016234776
  (0, 2965)	0.11605194227792441
  :	:
  (0, 3129)	0.05859233758688831
  (0, 8328)	0.03128679409255526
  (0, 559

In [16]:
df["clean_text"].iloc[0]

'marketing intern us ny new york food created groundbreaking award winning cooking site support connect celebrate home cooks give everything need one place top editorial business engineering team focused using technology find new better ways connect people around specific food interests offer superb highly curated information food cooking attract talented home cooks contributors country also publish well known professionals like mario batali gwyneth paltrow danny meyer partnerships whole foods market random house food named best food website james beard foundation iacp featured new york times npr pando daily techcrunch today show located chelsea new york city food fast growing james beard award winning online food community crowd sourced curated recipe hub currently interviewing full part time unpaid interns work small team editors executives developers new york city headquarters reproducing repackaging existing food content number partner sites huffington post yahoo buzzfeed various c

In [17]:
from sklearn.model_selection import train_test_split
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.2 , random_state = 42)

In [44]:
####### Baseline model

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score , f1_score , precision_score , confusion_matrix
model = XGBClassifier()
model.fit(X_train , y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test , y_pred))
print(f1_score(y_test , y_pred))

0.9818232662192393
0.7882736156351792


In [45]:
baseline_accuracy = 0.9818232662192393
baseline_f1 = 0.7882736156351792
baseline_cm = confusion_matrix(y_test , y_pred)
baseline_cm

array([[3390,    5],
       [  60,  121]])

In [23]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score , f1_score

param_grid = {
    'max_depth':[4,6],
    'learning_rate':[0.1,0.02,0.09],
    'n_estimators':[100,150],
    'scale_pos_weight':[10,15]
}

xgb = XGBClassifier()

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=6,
    scoring="f1",
    cv=3,
    verbose=1,
    n_jobs=-1
)

random_search.fit(X_train , y_train)

Fitting 3 folds for each of 6 candidates, totalling 18 fits


RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric=None, feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_constrain...
                                           max_delta_step=None, max_depth=None,
                                           max_leaves=None,
                                           min_child_weight=None, missing=nan,
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=6, n_jobs=-1,
                   param_distributions={'learning_rate': [0.1, 0.02, 0.09],
                                        'max_depth': [4, 6],
                                        'n_estimators': [100, 150],
                                        'scale_pos_weight': [10, 15]},
                   scoring='f1', verbose=1)

In [26]:
best_model = random_search.best_estimator_

In [27]:
y_pred = best_model.predict(X_test)

In [28]:
from sklearn.metrics import accuracy_score , f1_score , precision_score , recall_score
accuracy_score = accuracy_score(y_test , y_pred)
f1_score = f1_score(y_test , y_pred)

In [29]:
print(accuracy_score)
print(f1_score)

0.9762304250559284
0.7645429362880887


In [30]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print(cm)

[[3353   42]
 [  43  138]]


In [35]:
from sklearn.metrics import accuracy_score, f1_score

In [47]:
best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

tuned_accuracy = accuracy_score(y_test , y_pred)
tuned_f1 = f1_score(y_test , y_pred)
tuned_cm = confusion_matrix(y_test , y_pred)

print(tuned_accuracy)
print(tuned_f1)

0.9762304250559284
0.7645429362880887


In [48]:
import pandas as pd

comparison = pd.DataFrame({
    "Model": ["Baseline", "Tuned"],
    "Accuracy": [baseline_accuracy, tuned_accuracy],
    "F1 Score": [baseline_f1, tuned_f1],
    'confusion_matrix' : [baseline_cm , tuned_cm]

})

print(comparison)

      Model  Accuracy  F1 Score         confusion_matrix
0  Baseline  0.981823  0.788274   [[3390, 5], [60, 121]]
1     Tuned  0.976230  0.764543  [[3353, 42], [43, 138]]


Although the tuned model has slightly lower overall accuracy and F1 score, it significantly improves fraud detection by reducing false negatives from 60 to 43. Since detecting fraudulent job postings is the primary objective, the tuned model was selected as the final model.

Baseline → high precision

Tuned → better fraud detection

In [50]:
best_model = random_search.best_estimator_
best_model

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.09, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=150,
              n_jobs=None, num_parallel_tree=None, ...)

In [51]:
import joblib

joblib.dump(best_model, "tuned_xgboost_model.pkl")

['tuned_xgboost_model.pkl']

In [52]:
import joblib

joblib.dump(best_model, "/content/drive/MyDrive/tuned_xgboost_model.pkl")

['/content/drive/MyDrive/tuned_xgboost_model.pkl']

In [53]:
joblib.dump(tfidf, "/content/drive/MyDrive/tfidf_vectorizer.pkl")

['/content/drive/MyDrive/tfidf_vectorizer.pkl']